# pytorch training example

this notebook trains a small neural network to classify examples using two input features

In [1]:
# import pytorch and choose a device

import sys

import torch
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(123)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("python executable:", sys.executable)
print("torch version:", torch.__version__)
print("device:", device)

python executable: /Users/nabigchaudhry/Projects/build-llm-from-scratch/.venv/bin/python
torch version: 2.13.0
device: mps


In [2]:
# create the training and test data

training_features = torch.tensor(
    [
        [-2.0, -1.0],
        [-1.5, -1.8],
        [-1.0, -2.0],
        [1.0, 1.5],
        [1.5, 2.0],
        [2.0, 1.0],
    ],
    dtype=torch.float32,
)

training_labels = torch.tensor(
    [0, 0, 0, 1, 1, 1],
    dtype=torch.long,
)

test_features = torch.tensor(
    [
        [-1.25, -1.50],
        [1.25, 1.50],
    ],
    dtype=torch.float32,
)

test_labels = torch.tensor(
    [0, 1],
    dtype=torch.long,
)


print("training feature shape:", training_features.shape)
print("training label shape:", training_labels.shape)
print("test feature shape:", test_features.shape)
print("test label shape:", test_labels.shape)

training feature shape: torch.Size([6, 2])
training label shape: torch.Size([6])
test feature shape: torch.Size([2, 2])
test label shape: torch.Size([2])


In [3]:
# check the data

assert training_features.ndim == 2
assert training_labels.ndim == 1
assert training_features.shape[0] == training_labels.shape[0]
assert training_features.shape[1] == 2
assert training_features.dtype == torch.float32
assert training_labels.dtype == torch.long
assert torch.isfinite(training_features).all()
assert torch.all(
    (training_labels == 0)
    | (training_labels == 1)
)

print("all data checks passed")

all data checks passed


In [4]:
# define the dataset

class toy_dataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [5]:
# create datasets and data loaders

training_dataset = toy_dataset(training_features, training_labels)

test_dataset = toy_dataset(test_features, test_labels)

training_loader = DataLoader(training_dataset, batch_size=2, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [6]:
# inspect one batch

for batch_number, (batch_features, batch_labels) in enumerate(training_loader):
    print("batch number:", batch_number)
    print("batch features shape:", batch_features.shape)
    print("batch labels shape:", batch_labels.shape)
    print("batch features:", batch_features)
    print("batch labels:", batch_labels)
    break

batch number: 0
batch features shape: torch.Size([2, 2])
batch labels shape: torch.Size([2])
batch features: tensor([[-2.0000, -1.0000],
        [ 1.0000,  1.5000]])
batch labels: tensor([0, 1])


In [7]:
# define the neural network model

class toy_classifier(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(2, 4),
            torch.nn.ReLU(),
            torch.nn.Linear(4, 2),
        )

    def forward(self, x):
        return self.layers(x)

In [8]:
# create the model, loss function, and optimizer

torch.manual_seed(123)

model = toy_classifier()
model = model.to(device)

loss_function = torch.nn.CrossEntropyLoss()

learning_rate = 0.01

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(model)
print(next(model.parameters()).device,)

toy_classifier(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=2, bias=True)
  )
)
mps:0


In [9]:
# inspect the model's initial predictions

model.eval()

batch_features = batch_features.to(device)
batch_labels = batch_labels.to(device)

with torch.no_grad():
    initial_logits = model(batch_features)

    initial_probabilities = torch.softmax(initial_logits, dim=1)

    initial_predictions = torch.argmax(initial_probabilities, dim=1)

print("initial logits:", initial_logits)
print("initial probabilities:", initial_probabilities)
print("initial predictions:", initial_predictions)

initial logits: tensor([[-0.1813,  0.5476],
        [ 0.2416,  0.2140]], device='mps:0')
initial probabilities: tensor([[0.3255, 0.6745],
        [0.5069, 0.4931]], device='mps:0')
initial predictions: tensor([1, 0], device='mps:0')


In [10]:
# define an accuracy function

def calculate_accuracy(data_loader, model, device):

    model.eval()

    correct_predictions = 0
    total_examples = 0

    with torch.no_grad():
        for (batch_features, batch_labels) in data_loader:

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features)

            predictions = torch.argmax(logits, dim=1)

            correct_predictions += (predictions == batch_labels).sum().item()

            total_examples += batch_labels.shape[0]

    return correct_predictions / total_examples

In [11]:
# inspect accuracy before training

initial_training_accuracy = calculate_accuracy(training_loader, model, device)

initial_test_accuracy = calculate_accuracy(test_loader, model, device)

print("initial training accuracy:", initial_training_accuracy)
print("initial test accuracy:", initial_test_accuracy)

initial training accuracy: 0.0
initial test accuracy: 0.0


In [12]:
# train the model

number_of_epochs = 50

loss_history = []

for epoch in range(number_of_epochs):
    
    model.train()

    total_loss = 0.0

    for (batch_features, batch_labels) in training_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        logits = model(batch_features)

        loss = loss_function(logits, batch_labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(training_loader)

    loss_history.append(average_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        training_accuracy = calculate_accuracy(training_loader, model, device)
        test_accuracy = calculate_accuracy(test_loader, model, device)
    
        print(
                f"epoch {epoch + 1:02d} | "
                f"loss {average_loss:.4f} | "
                f"training accuracy "
                f"{training_accuracy:.2f} | "
                f"test accuracy "
                f"{test_accuracy:.2f}"
            )


epoch 01 | loss 0.8637 | training accuracy 0.00 | test accuracy 0.00
epoch 10 | loss 0.6245 | training accuracy 0.83 | test accuracy 1.00
epoch 20 | loss 0.4727 | training accuracy 1.00 | test accuracy 1.00
epoch 30 | loss 0.3536 | training accuracy 1.00 | test accuracy 1.00
epoch 40 | loss 0.2673 | training accuracy 1.00 | test accuracy 1.00
epoch 50 | loss 0.2021 | training accuracy 1.00 | test accuracy 1.00


In [13]:
# inspect the completed training

print("first recorded loss:", loss_history[0])
print("final recorded loss:", loss_history[-1])

final_training_accuracy = calculate_accuracy(training_loader, model, device)

final_test_accuracy = calculate_accuracy(test_loader, model, device)

print("final training accuracy:", final_training_accuracy)
print("final test accuracy:", final_test_accuracy)

first recorded loss: 0.8636674086252848
final recorded loss: 0.20206602911154428
final training accuracy: 1.0
final test accuracy: 1.0


In [14]:
# inspect the final test predictions

model.eval()

with torch.no_grad():
    
    test_features_device = test_features.to(device)
    test_logits = model(test_features_device)

    test_probabilities = torch.softmax(test_logits, dim=1)

    test_predictions = torch.argmax(test_probabilities, dim=1)

print("test logits:", test_logits)
print("test probabilities:", test_probabilities)
print("test predictions:", test_predictions)
print("correct test labels:", test_labels)

test logits: tensor([[ 1.0607, -0.4895],
        [-0.4449,  0.9279]], device='mps:0')
test probabilities: tensor([[0.8249, 0.1751],
        [0.2022, 0.7978]], device='mps:0')
test predictions: tensor([0, 1], device='mps:0')
correct test labels: tensor([0, 1])
